# Factor Analysis in Finance — Fama-French Model
## Sector ETF Exposures to Market, Size, and Value Factors
**Course:** MA2003B — Aplicación de métodos multivariados en ciencia de datos  
**Prof.:** Dr. Jonathan Montalvo Urquizo — October 2026

In [71]:
# Imports
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
from io import StringIO

In [72]:
# --- §1.2 Configuration ---
USE_FAMAFRENCH_3 = True
tickers = ["SOXX", "XLK", "XLY", "XLF"]
start_date = "2006-01-01"
end_date = "2026-10-01"

In [73]:
# --- §1.2 Step 1: Download prices and compute monthly simple returns ---
daily_data = yf.download(tickers, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)
daily_prices = daily_data["Close"]
monthly_prices = daily_prices.resample("ME").last()
returns = (monthly_prices / monthly_prices.shift(1)) - 1
returns = returns.dropna()
print(returns.head())

Ticker          SOXX       XLF       XLK       XLY
Date                                              
2006-02-28 -0.022128  0.021283  0.002315  0.007223
2006-03-31  0.003540  0.002898  0.022633  0.005884
2006-04-30  0.023470  0.043318 -0.013099  0.012775
2006-05-31 -0.094724 -0.037691 -0.064988 -0.016720
2006-06-30 -0.039947 -0.004626 -0.004895 -0.001750


In [74]:
from pathlib import Path

def clean_french_csv(raw_path: str, out_path: str) -> None:
    """Keep only the monthly block of the Ken French CSV and name the date column YYYYMM."""
    lines = Path(raw_path).read_text(encoding="utf-8", errors="ignore").splitlines()

    # 1) Header row = first line containing 'Mkt-RF'
    start = next(i for i, line in enumerate(lines) if "Mkt-RF" in line)

    # 2) Monthly block ends at the first blank line or the 'Annual Factors' marker
    end = next(
        (i for i in range(start + 1, len(lines))
         if not lines[i].strip() or "Annual" in lines[i]),
        len(lines),
    )

    block = lines[start:end]
    block[0] = "YYYYMM" + block[0]          # ',Mkt-RF,...' -> 'YYYYMM,Mkt-RF,...'
    Path(out_path).write_text("\n".join(block) + "\n", encoding="utf-8")


clean_french_csv("fama_french_factors.csv", "fama_french_factors_clean.csv")
DATA_FILE_PATH = "fama_french_factors_clean.csv"
factors_raw = pd.read_csv(DATA_FILE_PATH)
factors_raw.head()

,YYYYMM,Mkt-RF,SMB,HML,RMW,CMA,RF
0,196307,-0.39,-0.48,-0.84,0.64,-1.15,0.27
1,196308,5.08,-0.80,1.72,0.40,-0.38,0.25
2,196309,-1.57,-0.43,-0.02,-0.78,0.15,0.27
3,196310,2.54,-1.34,-0.03,2.79,-2.25,0.29
4,196311,-0.86,-0.85,1.78,-0.43,2.27,0.27


In [75]:
# --- §1.2 Step 2: Load Fama-French factors from cleaned CSV ---
DATA_FILE_PATH = "fama_french_factors_clean.csv"
factors_raw = pd.read_csv(DATA_FILE_PATH)

In [76]:
# --- §1.2 Step 3: Clean factors — parse dates, rescale to decimals ---
factors_raw.columns = factors_raw.columns.str.strip()
factors_raw["YYYYMM"] = factors_raw["YYYYMM"].astype(str).str.strip()
factors_raw["Date"] = pd.to_datetime(factors_raw["YYYYMM"], format="%Y%m")
factors = factors_raw.set_index("Date")

factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
factors = factors[factor_columns]
factors = factors.apply(pd.to_numeric, errors="coerce")
factors = factors / 100
factors = factors.dropna()

if USE_FAMAFRENCH_3:
    factors = factors[["Mkt-RF", "SMB", "HML", "RF"]]

print("Fama-French factors (first rows):")
print(factors.head())

Fama-French factors (first rows):
            Mkt-RF     SMB     HML      RF
Date                                      
1963-07-01 -0.0039 -0.0048 -0.0084  0.0027
1963-08-01  0.0508 -0.0080  0.0172  0.0025
1963-09-01 -0.0157 -0.0043 -0.0002  0.0027
1963-10-01  0.0254 -0.0134 -0.0003  0.0029
1963-11-01 -0.0086 -0.0085  0.0178  0.0027


In [77]:
# --- §1.2 Step 4: Inner join on common months ---
returns.index = returns.index.to_period("M")
factors.index = factors.index.to_period("M")

data = returns.join(factors, how="inner")
data = data.dropna()

print("Returns and factors (first rows):")
print(data.head())
print("\nFinal dimensions:", data.shape)
print("Common period:", data.index.min(), "to", data.index.max())

Returns and factors (first rows):
             SOXX       XLF       XLK       XLY  Mkt-RF     SMB     HML  \
Date                                                                      
2006-02 -0.022128  0.021283  0.002315  0.007223 -0.0030 -0.0046 -0.0031   
2006-03  0.003540  0.002898  0.022633  0.005884  0.0147  0.0334  0.0061   
2006-04  0.023470  0.043318 -0.013099  0.012775  0.0073 -0.0079  0.0226   
2006-05 -0.094724 -0.037691 -0.064988 -0.016720 -0.0356 -0.0290  0.0238   
2006-06 -0.039947 -0.004626 -0.004895 -0.001750 -0.0034 -0.0024  0.0085   

             RF  
Date             
2006-02  0.0034  
2006-03  0.0037  
2006-04  0.0036  
2006-05  0.0043  
2006-06  0.0040  

Final dimensions: (247, 8)
Common period: 2006-02 to 2026-08


In [78]:
# --- §1.2 Step 5: Compute excess returns (R - RF) for each ETF ---
for ticker in tickers:
    data[f"{ticker}_excess"] = data[ticker] - data["RF"]

excess_cols = [f"{t}_excess" for t in tickers]
print("Excess returns (first rows):")
print(data[excess_cols].head())

Excess returns (first rows):
         SOXX_excess  XLK_excess  XLY_excess  XLF_excess
Date                                                    
2006-02    -0.025528   -0.001085    0.003823    0.017883
2006-03    -0.000160    0.018933    0.002184   -0.000802
2006-04     0.019870   -0.016699    0.009175    0.039718
2006-05    -0.099024   -0.069288   -0.021020   -0.041991
2006-06    -0.043947   -0.008895   -0.005750   -0.008626
